# Included versus excluded participants

Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Compare categorical and continuous characteristics between participants with cluster assignments and those without them.

## Input

`files/tables/full_risk_factors_{coh}.xlsx`, default `coh = [3]` (alternative `[1, 2]`). Requires full-cohort records with IDX=0/1 for included participants and missing IDX for excluded participants. The export producing this file is inactive in the supplied risk-factor notebook; provide the existing workbook before running.

## Outputs

- `files/tables/incl_vs_exl_rr_cat_{coh}_statanalysis.xlsx`
- `files/tables/incl_vs_excl_cd_cont_{coh}_statanalysis.xlsx`



## 1. Setup and cohort input

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import seaborn as sns
import cohort_data_utils as ops
from scipy.stats import fisher_exact
from scipy.stats.contingency import relative_risk
from scipy.stats.contingency import odds_ratio

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2


In [ ]:
coh = [3]

from pathlib import Path

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "files" / "tables"
INPUT_PATH.mkdir(parents=True, exist_ok=True)

full_cohort = pd.read_excel(INPUT_PATH / f'full_risk_factors_{coh}.xlsx')
full_cohort['Psychiatric history'].value_counts()
full_cohort


In [ ]:
count_psy = full_cohort[full_cohort['IDX'].isin([0,1])]
count_psy['Psychiatric history'].value_counts()


## 2. Select cohort-specific variables

In [ ]:
if coh == [1, 2]:
    cont_var = ['Age',
        'Total amount of children',
        'Weight (gramm)','Stressful life events']

    categ_var = ['Highest degree of education',
                'Income', 'Baby blues', 'Marital status',
                'Breastfeeding T0', 'Breastfeeding T4',
                'Birth mode', 'Relocation to other ward',
                'Gender of the child',
                'Psychiatric diagnosis in previous pregancy', 'Familial psychiatric history',
                'Birth-related psychological and physical traumas', 'Psychiatric history binary', 'Diagnosis']
else:
    cont_var = ['Age','Total amount of children','Height (cm)', 'BMI before pregnancy',
        'BMI after birth','BMI 3 months after birth','Weight (gramm)','Stressful life events',
        'RLS_Wert','PMS_value','Ablehnung T1','Ablehnung T2','Ablehnung T3','Ablehnung T4','Severity of baby blues',
        'Empathy','System','Mutter Care','Mutter Overprotection','Vater Care','Vater Overprotection']

    categ_var = ['Highest degree of education', 'Income',
               'Support at home t0', 'Support at home t4', 'PMS Severity',
               'Baby blues', 'Marital status', 'Breastfeeding T0',
               'Breastfeeding T4', 'in vitro fertilization2', 'Birth mode',
               'Complications at birth', 'Complication during pregancy', 'Relocation to other ward',
               'Gender of the child', 'Psychiatric diagnosis in previous pregancy', 'Familial psychiatric history',
               'RLS', 'Birth-related psychological and physical traumas', 'Psychiatric history binary', 'Diagnosis']

    exlude_cont = ['Age', 'Total amount of children', 'Weight (gramm)', 'Stressful life events']
    cont_only_val = [x for x in cont_var if x not in exlude_cont]

    exclude_cat = ['Highest degree of education',  'Income', 'Baby blues', 'Marital status', 'Breastfeeding T0', 'Breastfeeding T4',
                'Birth mode', 'Relocation to other ward','Gender of the child', 'Psychiatric diagnosis in previous pregancy', 'Familial psychiatric history',
                'Birth-related psychological and physical traumas', 'Psychiatric history binary']
    cat_only_val = [x for x in categ_var if x not in exclude_cat]


In [ ]:
full_cohort['IDX'].value_counts()


## 3. Categorical comparisons

Recode original IDX=0/1 to included=0 and missing IDX to excluded=1. Relative risk is the proportion with factor value 1 among excluded participants divided by that among included participants, with a 95% SciPy interval. Use chi-square when the helper allows at most one expected count below five; otherwise use Fisher’s exact test. The test extracts only 0/1 rows while denominators use all nonmissing categories, so inputs must have the intended binary coding. Variables lacking required binary rows are skipped; Diagnosis is not tested as a multi-category variable. Stars use p-values rounded to three decimals.

In [ ]:
selected = full_cohort[categ_var]
selected['IDX'] = full_cohort['IDX']
# Collapse both assigned clusters into the included group.
selected["IDX"] = selected["IDX"].replace(1, 0)
selected["IDX"] = selected["IDX"].fillna(1)
print(selected["IDX"].value_counts())

selected['Diagnosis'] = full_cohort['Diagnosis']
for col in categ_var:
    selected[col] = selected[col].replace({0: 0.0, 1: 1.0})

filter_idx_0 = selected[(selected['IDX'] == 0)]
filter_idx_1_ppd = selected[(selected['IDX'] == 1)]
result = pd.concat([filter_idx_0, filter_idx_1_ppd])

p_values = []
contingency_tables = []
validities = []
rr_values = []

summary_data = []

for var in categ_var:
    selected_df = result
    contingency_table = pd.crosstab(selected_df[var], selected_df['IDX'])
    print(var)
    print(contingency_table)

    exposed_total = contingency_table[1].sum()
    control_total = contingency_table[0].sum()

    exposed_cases = (filter_idx_1_ppd[var] == 1).sum()
    control_cases = (filter_idx_0[var] == 1).sum()
    pass

    try:
        a = contingency_table.loc[1.0, 1]
        b = contingency_table.loc[1.0, 0]
        c = contingency_table.loc[0.0, 1]
        d = contingency_table.loc[0.0, 0]
    except KeyError:
        print(f"Issue in contingency table for variable '{var}' — skipping.")
        continue

    odds_table = [[a, b],
                [c, d]]

    is_valid, chi2, p, dof, expected = ops.check_contingency_table(odds_table)
    if is_valid:
        validities.append(is_valid)
        p_values.append(p)
    else:
        stat, p = fisher_exact(odds_table)
        p_values.append(p)

    rr = relative_risk(
        exposed_cases=exposed_cases,
        exposed_total=exposed_total,
        control_cases=control_cases,
        control_total=control_total
    )
    odds = odds_ratio(odds_table)

    summary_data.append({
    'Variable': var,
    'Exposed population': exposed_total,
    'Exposed events': exposed_cases,
    'Unexposed population': control_total,
    'Unexposed events': control_cases,
    'Validity': is_valid,
    'P-value': round(p,3),
    'RR': round(rr.relative_risk,3),
    'CI Lower': round(rr.confidence_interval(confidence_level=0.95).low,3),
    'CI Upper': round(rr.confidence_interval(confidence_level=0.95).high,3)

    })
summary_df = pd.DataFrame(summary_data)

def significance_marker(row):
    """Mark rounded, unadjusted p-values below 0.05."""
    if row['P-value'] < 0.05:
        return '*'
    else:
        return ''

summary_df['Significance'] = summary_df.apply(significance_marker, axis=1)

summary_df


### Validation-only correction

Apply BH correction to the selected validation-only categorical variables using their rounded p-values. The final display references `only_val_pvals` outside the cohort condition, so a fresh discovery run can fail here; the supplied code is preserved.

In [ ]:
from statsmodels.stats.multitest import multipletests
if coh == [3]:

    only_val_pvals = summary_df[summary_df["Variable"].isin(cat_only_val)][['Variable', 'P-value']].dropna()

    rejected, pvals_corrected, _, _ = multipletests(only_val_pvals['P-value'], alpha=0.05, method='fdr_bh')

    only_val_pvals['FDR_corrected_p'] = pvals_corrected
    only_val_pvals['FDR_significant'] = rejected
    significant_vars_val = only_val_pvals[only_val_pvals['FDR_significant']]

    print("Significant (FDR-corrected) variables only in validation cohort:")
    print(significant_vars_val[['Variable', 'P-value', 'FDR_corrected_p']])

only_val_pvals


## 4. Continuous comparisons

Cohen’s d is `(excluded mean − included mean) / pooled sample SD`. Independently bootstrap each group 10,000 times using a generator reset to seed 42 and report the 2.5th–97.5th percentile interval.

In [ ]:
def cohen_d_bootstrap(group1, group2, n_bootstrap = 10000, alpha=0.05, random_state=42):

    """Return group-2 minus group-1 Cohen’s d and its percentile bootstrap interval."""
    rng = np.random.default_rng(random_state)

    mean1, mean2 = np.mean(group1), np.mean(group2)
    std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
    pooled_std = np.sqrt(((len(group1) - 1)*std1**2 + (len(group2) - 1)*std2**2) / (len(group1) + len(group2) - 2))
    mean_diff = mean2 - mean1
    cohen_d = mean_diff / pooled_std

    boot_d = []
    for _ in range(n_bootstrap):
        sample1 = rng.choice(group1, size=len(group1), replace=True)
        sample2 = rng.choice(group2, size=len(group2), replace=True)
        m1, m2 = np.mean(sample1), np.mean(sample2)
        s1, s2 = np.std(sample1, ddof=1), np.std(sample2, ddof=1)

        pooled = np.sqrt(((len(sample1) - 1)*s1**2 + (len(sample2) - 1)*s2**2) / (len(sample1) + len(sample2) - 2))
        d = (m2 - m1) / pooled
        boot_d.append(d)

    ci_lower = np.percentile(boot_d, 100 * alpha / 2)

    ci_upper = np.percentile(boot_d, 100 * (1 - alpha / 2))

    return cohen_d, ci_lower, ci_upper


In [ ]:
selected_cont = full_cohort[cont_var]
selected_cont['Diagnosis'] = full_cohort['Diagnosis']
selected_cont['IDX'] = full_cohort['IDX']
selected_cont["IDX"] = selected_cont["IDX"].replace(1, 0)
selected_cont["IDX"] = selected_cont["IDX"].fillna(1)
print(selected_cont["IDX"].value_counts())

selected_cont


Coerce scores to numeric and drop missing values within each group. Run two-group Kruskal–Wallis tests and calculate effect sizes. The size check uses pre-drop row counts; small nonmissing samples and zero variance are not fully handled. The legacy count labels “Healthy” and “Healthy at risk” actually represent included and excluded participants here.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

filter_idx_0_cont = selected_cont[(selected_cont['IDX'] == 0)]
filter_idx_1_ppd_cont = selected_cont[(selected_cont['IDX'] == 1)]

result_cont = pd.concat([filter_idx_0_cont, filter_idx_1_ppd_cont])

significant_var = []
p_values = []
cohens_d_values = []

summary_data_cont = []

for var in cont_var:

    filter_idx_0_cont[var] = pd.to_numeric(filter_idx_0_cont[var], errors='coerce')
    filter_idx_1_ppd_cont[var] = pd.to_numeric(filter_idx_1_ppd_cont[var], errors='coerce')

    data1 = filter_idx_0_cont[var].dropna()
    data2 = filter_idx_1_ppd_cont[var].dropna()

    print(var)

    if len(filter_idx_0_cont[var]) < 2 or len(filter_idx_1_ppd_cont[var]) < 2:
        print(f"Skipping {var} due to insufficient data")
        continue

    Kruskal_result = stats.kruskal(data1,
                                data2)
    print('p:', Kruskal_result.pvalue)
    p_values.append(Kruskal_result.pvalue)

    cohens_d, ci_lower, ci_upper = cohen_d_bootstrap(data1, data2)

    h = filter_idx_0_cont[var].count()
    hr = filter_idx_1_ppd_cont[var].count()

    summary_data_cont.append({
        'Variable': var,
        'Cohen\'s d': round(cohens_d,3),
        'CI Lower': round(ci_lower,3),
        'CI Upper': round(ci_upper,3),
        'P-value': round(Kruskal_result.pvalue,3),
        'Healthy': h,
        'Healthy at risk': hr,

    })

valid_p_values = [p for p in p_values if not np.isnan(p)]
valid_indices = [i for i, p in enumerate(p_values) if not np.isnan(p)]

summary_df_cont = pd.DataFrame(summary_data_cont)
summary_df_cont['P-value'] = pd.to_numeric(summary_df_cont['P-value'], errors='coerce')
def significance_marker(row):
    """Mark rounded, unadjusted p-values below 0.05."""
    if row['P-value'] < 0.05:
        return '*'
    else:
        return ''

summary_df_cont['Significance'] = summary_df_cont.apply(significance_marker, axis=1)

summary_df_cont


Apply BH correction separately to the validation-only continuous variables. Common variables are not included in this correction; the main tables retain unadjusted p-values and stars.

In [ ]:
if coh == [3]:

    only_val_pvals = summary_df_cont[summary_df_cont["Variable"].isin(cont_only_val)][['Variable', 'P-value']].dropna()

    rejected, pvals_corrected, _, _ = multipletests(only_val_pvals['P-value'], alpha=0.05, method='fdr_bh')

    only_val_pvals['FDR_corrected_p'] = pvals_corrected
    only_val_pvals['FDR_significant'] = rejected
    significant_vars_val = only_val_pvals[only_val_pvals['FDR_significant']]

    print("Significant (FDR-corrected) variables only in validation cohort:")
    print(significant_vars_val[['Variable', 'P-value', 'FDR_corrected_p']])
    only_val_pvals
only_val_pvals


## 5. Label and export summaries

In [ ]:
rename_dict = {
    "Highest degree of education": "Education",
    "Income": "Annual income, thousand €",
    "Baby blues": "Baby Blues",
    "Marital status": "Married",
    "Breastfeeding T0": "Breastfeeding at childbirth",
    "Breastfeeding T4": "Breastfeeding at 12 weeks pp",
    "Gender of the child": "Gender of the baby",
    "Birth-related psychological and physical traumas": "Subjective experience of traumatic childbirth",
    "Support at home t0": "Support at home at childbirth",
    "Support at home t4": "Support at home at 12 weeks pp",
    "Psychiatric history binary":"Psychiatric history",
    "Age": "Age, years",
    "Total amount of children": "Total number of children",
    "Stressful life events": "Stressful life events (number)",
    "Weight (gramm)": "Birthweight of the child, g",
    "Height (cm)": "Height, cm",
    "Rejection T0": "Rejection towards the baby T0",
    "Rejection T1": "Rejection towards the baby T1",
    "Rejection T2": "Rejection towards the baby T2",
    "Rejection T3": "Rejection towards the baby T3",
    "Mother care": "Care from own mother",
    "Father care": "Care from own father",
    "Mother overprotection": "Overprotection by own mother",
    "Father overprotection": "Overprotection by own father"
}

def rename_values_in_column(df, column, mapping, df_name):
    """Rename mapped display values in place and print changes."""
    if column not in df.columns:
        print(f"Column '{column}' not found in {df_name} DataFrame.")
        return df

    renamed = []
    for old_val in df[column]:
        if old_val in mapping:
            renamed.append(mapping[old_val])
            print(f"{df_name}: {old_val} → {mapping[old_val]}")
        else:
            renamed.append(old_val)
    df[column] = renamed
    return df

if coh == [1, 2]:
    summary_df_cat = rename_values_in_column(summary_df, "Variable", rename_dict, "Discovery")
    summary_df_cont = rename_values_in_column(summary_df_cont, "Variable", rename_dict, "Discovery")
else:
    summary_df_cat = rename_values_in_column(summary_df, "Variable", rename_dict, "Validation")
    summary_df_cont = rename_values_in_column(summary_df_cont, "Variable", rename_dict, "Discovery")


In [ ]:
summary_df_cont


In [ ]:
summary_df_cat.to_excel(INPUT_PATH / f'incl_vs_exl_rr_cat_{coh}_statanalysis.xlsx', index=False)
summary_df_cont.to_excel(INPUT_PATH / f'incl_vs_excl_cd_cont_{coh}_statanalysis.xlsx', index=False)
